# Step 3: Instruction Tuning Format

Convert curated data into the format expected by your target model family.

**What this notebook covers:**
- Building chat conversations from NL→SQL pairs
- Applying model-specific templates (Llama, Mistral, Phi)
- Tokenization stats and max-length validation
- Exporting formatted dataset for training

# ⚠️ IMPORTANT - READ BEFORE RUNNING

**This notebook will RE-FORMAT DATA and OVERWRITE existing files.**

## Purpose
This is an **educational walkthrough** demonstrating how the formatting pipeline works. It will:
- Re-format curated data from scratch
- OVERWRITE `data/formatted/*.jsonl` if they exist

## When to Use This Notebook
✅ **LEARNING**: Understanding how instruction formatting works  
✅ **DEVELOPMENT**: Testing new format templates  
✅ **FRESH START**: Starting completely from scratch

## When NOT to Use This Notebook
❌ **EVALUATION**: You have already formatted data and want to evaluate your trained model  
❌ **PRODUCTION**: You want to use existing formatted data  
❌ **COMPARISON**: You want to compare teacher vs student models

## What You Should Run Instead
If you have completed training and want to evaluate your model, run:
- `notebooks/07_comparison_glm.ipynb` (if you have GLM API)
- `notebooks/07_comparison_anthropic.ipynb` (if you have Anthropic API)

See [docs/notebook-guide.md](docs/notebook-guide.md) for complete guidance.

---

# Step 3: Instruction Tuning Format

Convert curated data into the format expected by your target model family.

**What this notebook covers:**
- Building chat conversations from NL→SQL pairs
- Applying model-specific templates (Llama, Mistral, Phi)
- Tokenization stats and max-length validation
- Exporting formatted dataset for training

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from src.format.templates import ChatTemplate
from src.format.families import get_family
from src.format.tokenize import TokenStats, validate_max_length
from transformers import AutoTokenizer

In [ ]:
# Load curated training data
with open('data/curated/train.jsonl') as f:
    train_data = [json.loads(line) for line in f]
print(f'Loaded {len(train_data)} training examples')

In [ ]:
# Build conversations with system prompt
template = ChatTemplate()
conversations = [template.build_conversation(ex) for ex in train_data]
conversations[0]

In [ ]:
# Apply Llama 3.1 chat template
llama = get_family('llama')
formatted = [llama.apply_template(conv) for conv in conversations]
print(formatted[0][:500])

In [ ]:
# Check token length distribution
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3.1-8B-Instruct')

stats = TokenStats(tokenizer)
stats.analyze(formatted)
stats.summary()

In [ ]:
# Validate max length and filter
valid, too_long = validate_max_length(formatted, tokenizer, max_len=2048)
print(f'Valid: {len(valid)}, Too long: {len(too_long)}')

In [ ]:
# Convert to ShareGPT format and save
sharegpt = template.to_sharegpt(train_data)
with open('data/formatted/train.jsonl', 'w') as f:
    for ex in sharegpt:
        f.write(json.dumps(ex) + '\n')
print(f'Saved {len(sharegpt)} formatted examples')